# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_03 — Square-Root Market Impact Model

### Research question

How can we model, calibrate, stress-test, and govern the square-root market impact law used to estimate the cost of trading large orders?

This notebook follows:

```text
06_01_almgren_chriss_optimal_execution
06_02_dynamic_programming_execution
06_03_square_root_market_impact_model
```

The previous notebooks built execution schedules and dynamic execution policies. This notebook focuses on a central empirical model used in execution and capacity research:

$$
Impact \approx Y \sigma \sqrt{\frac{Q}{V}}
$$

where:

- $Q$: order size or metaorder notional/shares;
- $V$: daily volume or relevant volume benchmark;
- $\sigma$: volatility;
- $Y$: impact coefficient.

It covers:

1. market impact taxonomy;
2. temporary versus permanent impact;
3. square-root impact law;
4. participation rate;
5. synthetic metaorder generation;
6. realised impact measurement;
7. log-log calibration;
8. estimating exponent $\beta$;
9. comparing linear, square-root, and power-law models;
10. regime-conditioned impact;
11. side-conditioned impact;
12. intraday volume effects;
13. impact decay and transient impact;
14. permanent impact proxy;
15. bootstrap confidence intervals;
16. prediction intervals;
17. capacity curves;
18. execution schedule impact;
19. stress testing;
20. governance flags;
21. saved outputs and manifest.

The key idea:

> Market impact is usually concave: doubling trade size does not usually double impact, but it still hurts — and it hurts more when volatility is high and liquidity is thin.

## 1. Why square-root impact matters

In backtests, transaction cost is often simplified to:

$$
cost = turnover \times fixed\_bps
$$

That is not enough for capacity.

Large trades move prices.

The square-root law approximates how expected impact grows with trade size:

$$
I(Q) = Y\sigma\sqrt{\frac{Q}{V}}
$$

Important consequences:

1. impact rises with volatility;
2. impact falls with liquidity;
3. impact is concave in size;
4. capacity is nonlinear;
5. high-Sharpe strategies can fail once scaled.

## 2. Impact taxonomy

### Spread cost

Crossing the bid/ask spread.

### Temporary impact

Immediate price concession while executing.

### Permanent impact

Lasting price change after the trade, often associated with information content.

### Transient impact

Impact decays after execution but not instantly.

### Opportunity cost

Cost of not completing the order.

This notebook focuses on estimating realised market impact from synthetic metaorders.

## 3. The square-root impact model

A common normalised specification:

$$
\frac{I}{\sigma} = Y \Big(\frac{Q}{V}\Big)^\beta
$$

For the square-root law:

$$
\beta = \frac{1}{2}
$$

Taking logs:

$$
\begin{aligned}
\log\Big(\frac{I}{\sigma}\Big) &= \log(Y) \\
&\quad + \beta \log\Big(\frac{Q}{V}\Big) \\
&\quad + \epsilon
\end{aligned}
$$

This makes calibration simple with regression, but the details matter:

- zero or negative impact observations need handling;
- noise creates bias;
- regimes matter;
- small trades can be dominated by spread/noise;
- large trades can be constrained or censored.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class SquareRootImpactConfig:
    n_days: int = 700
    metaorders_per_day_mean: float = 8.0
    seed: int = 42
    base_price: float = 50.0
    annualisation: int = 252
    true_y: float = 0.75
    true_beta: float = 0.50
    permanent_fraction: float = 0.35
    decay_half_life_intervals: float = 6.0
    min_participation: float = 0.001
    max_participation: float = 0.30
    noise_bps: float = 3.0
    spread_bps_base: float = 1.8
    bootstrap_samples: int = 1_000
    stress_y_multiplier: float = 1.75
    stress_vol_multiplier: float = 2.00
    output_subdir: str = "square_root_market_impact_model_v1"

config = SquareRootImpactConfig()
config

## 5. Simulate daily market state

We simulate:

- daily volume;
- volatility;
- spread;
- liquidity regime;
- market return.

This creates an environment where impact depends on volatility and liquidity.

In [ ]:
def simulate_daily_market_state(config: SquareRootImpactConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2020-01-01", periods=config.n_days)

    regime = np.zeros(config.n_days, dtype=int)
    transition = np.array([
        [0.985, 0.015],
        [0.060, 0.940],
    ])

    for t in range(1, config.n_days):
        regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

    base_adv = 20_000_000.0
    base_vol_daily = 0.020

    adv = base_adv * rng.lognormal(0.0, 0.35, config.n_days) * np.where(regime == 0, 1.0, 0.55)
    vol = base_vol_daily * rng.lognormal(0.0, 0.20, config.n_days) * np.where(regime == 0, 1.0, 2.0)
    spread = config.spread_bps_base * rng.lognormal(0.0, 0.20, config.n_days) * np.where(regime == 0, 1.0, 2.5)

    market_return = rng.standard_t(df=6, size=config.n_days) * vol

    market_state = pd.DataFrame({
        "date": dates,
        "regime": regime,
        "regime_name": np.where(regime == 0, "normal_liquidity", "stress_liquidity"),
        "adv_shares": adv,
        "daily_vol": vol,
        "spread_bps": spread,
        "market_return": market_return,
    }).set_index("date")

    market_state["price"] = config.base_price * np.exp(np.log1p(market_state["market_return"]).cumsum())

    return market_state

market_state = simulate_daily_market_state(config)

market_state.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(market_state.index, market_state["price"])
plt.title("Synthetic Price Path")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(market_state.index, market_state["daily_vol"], label="daily vol")
plt.plot(market_state.index, market_state["adv_shares"] / market_state["adv_shares"].median() * market_state["daily_vol"].median(), label="scaled ADV")
plt.title("Volatility and Scaled Liquidity")
plt.xlabel("Date")
plt.legend()
plt.show()

## 6. Generate synthetic metaorders

Each metaorder has:

- date;
- side;
- size $Q$;
- daily volume $V$;
- participation $Q/V$;
- volatility $\sigma$;
- regime;
- realised impact.

The true data-generating process uses a power law:

$$
\begin{aligned}
I &= side \times Y\sigma \Big(\frac{Q}{V}\Big)^\beta \\
&\quad + noise
\end{aligned}
$$

Observed realised impact is noisy and can be affected by market drift.

In [ ]:
def simulate_metaorders(config: SquareRootImpactConfig, market_state: pd.DataFrame):
    rng = np.random.default_rng(config.seed + 123)
    rows = []

    order_id = 0
    for date, row in market_state.iterrows():
        n_orders = rng.poisson(config.metaorders_per_day_mean)
        n_orders = max(n_orders, 1)

        for _ in range(n_orders):
            side = rng.choice([-1, 1])
            participation = np.exp(rng.uniform(np.log(config.min_participation), np.log(config.max_participation)))
            q_shares = participation * row["adv_shares"]

            regime_multiplier = config.stress_y_multiplier if row["regime"] == 1 else 1.0
            true_y = config.true_y * regime_multiplier
            true_beta = config.true_beta

            clean_abs_impact_return = true_y * row["daily_vol"] * participation ** true_beta
            signed_clean_impact_return = side * clean_abs_impact_return

            drift_noise = rng.normal(0.0, row["daily_vol"] * 0.15)
            measurement_noise = rng.normal(0.0, config.noise_bps / 10000.0)

            realised_signed_impact_return = signed_clean_impact_return + drift_noise + measurement_noise
            realised_abs_impact_return = side * realised_signed_impact_return

            permanent_abs_return = config.permanent_fraction * clean_abs_impact_return + rng.normal(0.0, row["daily_vol"] * 0.05)
            temporary_abs_return = clean_abs_impact_return - permanent_abs_return

            rows.append({
                "order_id": order_id,
                "date": date,
                "side": side,
                "side_name": "BUY" if side > 0 else "SELL",
                "q_shares": q_shares,
                "adv_shares": row["adv_shares"],
                "participation": participation,
                "daily_vol": row["daily_vol"],
                "spread_bps": row["spread_bps"],
                "price": row["price"],
                "regime": int(row["regime"]),
                "regime_name": row["regime_name"],
                "true_y": true_y,
                "true_beta": true_beta,
                "clean_abs_impact_return": clean_abs_impact_return,
                "realised_signed_impact_return": realised_signed_impact_return,
                "realised_abs_impact_return": realised_abs_impact_return,
                "permanent_abs_return": permanent_abs_return,
                "temporary_abs_return": temporary_abs_return,
                "notional": q_shares * row["price"],
                "impact_bps": realised_abs_impact_return * 10000.0,
                "clean_impact_bps": clean_abs_impact_return * 10000.0,
            })
            order_id += 1

    metaorders = pd.DataFrame(rows)
    metaorders["normalised_abs_impact"] = metaorders["realised_abs_impact_return"] / metaorders["daily_vol"]
    metaorders["clean_normalised_impact"] = metaorders["clean_abs_impact_return"] / metaorders["daily_vol"]

    return metaorders

metaorders = simulate_metaorders(config, market_state)

metaorders.head(), metaorders.shape

## 7. Basic metaorder diagnostics

In [ ]:
metaorder_summary = pd.DataFrame([{
    "n_metaorders": len(metaorders),
    "mean_participation": metaorders["participation"].mean(),
    "p95_participation": metaorders["participation"].quantile(0.95),
    "mean_impact_bps": metaorders["impact_bps"].mean(),
    "median_impact_bps": metaorders["impact_bps"].median(),
    "p95_impact_bps": metaorders["impact_bps"].quantile(0.95),
    "mean_daily_vol": metaorders["daily_vol"].mean(),
    "mean_notional": metaorders["notional"].mean(),
}])

metaorder_summary.T

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(metaorders["participation"], metaorders["impact_bps"], alpha=0.25)
plt.xscale("log")
plt.axhline(0, linestyle="--")
plt.title("Observed Impact vs Participation")
plt.xlabel("Participation Q/V")
plt.ylabel("Realised impact, bps")
plt.show()

plt.figure(figsize=(10, 6))
plt.scatter(metaorders["participation"], metaorders["clean_impact_bps"], alpha=0.25)
plt.xscale("log")
plt.title("Clean True Impact vs Participation")
plt.xlabel("Participation Q/V")
plt.ylabel("Clean impact, bps")
plt.show()

## 8. Calibration sample filtering

Observed impacts can be negative because of noise and drift.

For log-log calibration, we need positive normalised impact:

$$
\log(I/\sigma)
$$

So we create a calibration sample using:

- positive observed impact;
- participation inside bounds;
- finite volatility;
- nonzero volume.

This filtering step is important and should be reported.

In [ ]:
def make_calibration_sample(metaorders):
    sample = metaorders.copy()
    sample = sample[
        (sample["normalised_abs_impact"] > 0)
        & (sample["participation"] > 0)
        & np.isfinite(sample["normalised_abs_impact"])
        & np.isfinite(sample["participation"])
        & np.isfinite(sample["daily_vol"])
    ].copy()

    sample["log_participation"] = np.log(sample["participation"])
    sample["log_normalised_impact"] = np.log(sample["normalised_abs_impact"])
    sample["log_clean_normalised_impact"] = np.log(sample["clean_normalised_impact"])
    return sample

calibration_sample = make_calibration_sample(metaorders)

calibration_report = pd.DataFrame([{
    "n_raw_metaorders": len(metaorders),
    "n_calibration_sample": len(calibration_sample),
    "retention_rate": len(calibration_sample) / len(metaorders),
    "negative_or_zero_observed_impact_count": int((metaorders["normalised_abs_impact"] <= 0).sum()),
}])

calibration_report

## 9. Log-log power-law calibration

Estimate:

$$
\log(I/\sigma) = \alpha + \beta\log(Q/V)+\epsilon
$$

Then:

$$
Y = e^\alpha
$$

Square-root impact predicts:

$$
\beta \approx 0.5
$$

In [ ]:
def ols_fit(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    beta = np.linalg.pinv(X.T @ X) @ X.T @ y
    fitted = X @ beta
    residual = y - fitted

    n, k = X.shape
    s2 = (residual @ residual) / max(n - k, 1)
    cov = s2 * np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(cov))

    ss_res = residual @ residual
    ss_tot = ((y - y.mean()) @ (y - y.mean()))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return {
        "coef": beta,
        "se": se,
        "fitted": fitted,
        "residual": residual,
        "r2": r2,
        "sigma2": s2,
    }

def calibrate_power_law(sample):
    X = np.column_stack([np.ones(len(sample)), sample["log_participation"].to_numpy()])
    y = sample["log_normalised_impact"].to_numpy()

    fit = ols_fit(X, y)
    alpha, beta = fit["coef"]
    alpha_se, beta_se = fit["se"]

    return {
        "alpha": alpha,
        "beta": beta,
        "Y": float(np.exp(alpha)),
        "alpha_se": alpha_se,
        "beta_se": beta_se,
        "r2": fit["r2"],
        "n": len(sample),
        "residual_std": float(np.sqrt(fit["sigma2"])),
    }, fit

power_law_params, power_law_fit = calibrate_power_law(calibration_sample)

pd.Series(power_law_params)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(calibration_sample["log_participation"], calibration_sample["log_normalised_impact"], alpha=0.25, label="observed")
x_grid = np.linspace(calibration_sample["log_participation"].min(), calibration_sample["log_participation"].max(), 100)
y_fit = power_law_params["alpha"] + power_law_params["beta"] * x_grid
plt.plot(x_grid, y_fit, linewidth=3, label="power-law fit")
plt.title("Log-Log Impact Calibration")
plt.xlabel("log(Q/V)")
plt.ylabel("log(I / sigma)")
plt.legend()
plt.show()

## 10. Compare linear, square-root, and free power-law models

Models:

### Linear impact

$$
I/\sigma = Y_1(Q/V)
$$

### Square-root impact

$$
I/\sigma = Y_{1/2}\sqrt{Q/V}
$$

### Free power law

$$
I/\sigma = Y_\beta(Q/V)^\beta
$$

We compare fit quality using RMSE and out-of-sample split.

In [ ]:
def fit_fixed_exponent(sample, exponent):
    x = sample["participation"].to_numpy()
    y = sample["normalised_abs_impact"].to_numpy()
    basis = x ** exponent

    y_coef = np.sum(basis * y) / max(np.sum(basis**2), 1e-18)
    fitted = y_coef * basis
    residual = y - fitted

    return {
        "model": f"fixed_beta_{exponent:.2f}",
        "Y": y_coef,
        "beta": exponent,
        "rmse": float(np.sqrt(np.mean(residual**2))),
        "mae": float(np.mean(np.abs(residual))),
        "bias": float(np.mean(residual)),
    }, fitted

def fit_free_power_predict(sample):
    params, fit = calibrate_power_law(sample)
    fitted = params["Y"] * sample["participation"].to_numpy() ** params["beta"]
    residual = sample["normalised_abs_impact"].to_numpy() - fitted
    out = {
        "model": "free_power_law",
        "Y": params["Y"],
        "beta": params["beta"],
        "rmse": float(np.sqrt(np.mean(residual**2))),
        "mae": float(np.mean(np.abs(residual))),
        "bias": float(np.mean(residual)),
        "r2_log": params["r2"],
    }
    return out, fitted

model_rows = []
fitted_store = {}

for exponent, name in [(1.0, "linear"), (0.5, "square_root")]:
    row, fitted = fit_fixed_exponent(calibration_sample, exponent)
    row["model"] = name
    model_rows.append(row)
    fitted_store[name] = fitted

row, fitted = fit_free_power_predict(calibration_sample)
model_rows.append(row)
fitted_store["free_power_law"] = fitted

model_comparison = pd.DataFrame(model_rows).sort_values("rmse")

model_comparison

## 11. Out-of-sample model comparison

Use a chronological split:

- first 70% calibration;
- final 30% validation.

This prevents judging models only in sample.

In [ ]:
def chronological_train_test(sample, train_frac=0.70):
    sample_sorted = sample.sort_values("date").reset_index(drop=True)
    split = int(len(sample_sorted) * train_frac)
    train = sample_sorted.iloc[:split].copy()
    test = sample_sorted.iloc[split:].copy()
    return train, test

train_sample, test_sample = chronological_train_test(calibration_sample, train_frac=0.70)

def evaluate_model_on_test(train, test):
    rows = []

    # Linear and square-root.
    for exponent, name in [(1.0, "linear"), (0.5, "square_root")]:
        fit_row, _ = fit_fixed_exponent(train, exponent)
        y_pred = fit_row["Y"] * test["participation"].to_numpy() ** exponent
        y_true = test["normalised_abs_impact"].to_numpy()
        residual = y_true - y_pred
        rows.append({
            "model": name,
            "train_Y": fit_row["Y"],
            "train_beta": exponent,
            "test_rmse": float(np.sqrt(np.mean(residual**2))),
            "test_mae": float(np.mean(np.abs(residual))),
            "test_bias": float(np.mean(residual)),
        })

    params, _ = calibrate_power_law(train)
    y_pred = params["Y"] * test["participation"].to_numpy() ** params["beta"]
    y_true = test["normalised_abs_impact"].to_numpy()
    residual = y_true - y_pred
    rows.append({
        "model": "free_power_law",
        "train_Y": params["Y"],
        "train_beta": params["beta"],
        "test_rmse": float(np.sqrt(np.mean(residual**2))),
        "test_mae": float(np.mean(np.abs(residual))),
        "test_bias": float(np.mean(residual)),
    })

    return pd.DataFrame(rows).sort_values("test_rmse")

oos_model_comparison = evaluate_model_on_test(train_sample, test_sample)

oos_model_comparison

## 12. Regime-conditioned calibration

Impact usually gets worse in stress:

$$
Y_{stress} > Y_{normal}
$$

We calibrate impact separately by regime.

In [ ]:
regime_calibration_rows = []

for regime_name, group in calibration_sample.groupby("regime_name"):
    params, _ = calibrate_power_law(group)
    row = {
        "regime_name": regime_name,
        **params,
        "mean_participation": group["participation"].mean(),
        "mean_impact_bps": group["impact_bps"].mean(),
    }
    regime_calibration_rows.append(row)

regime_calibration = pd.DataFrame(regime_calibration_rows).sort_values("Y", ascending=False)

regime_calibration

In [ ]:
plt.figure(figsize=(9, 5))
plt.barh(regime_calibration["regime_name"], regime_calibration["Y"])
plt.title("Calibrated Impact Coefficient Y by Regime")
plt.xlabel("Y")
plt.ylabel("Regime")
plt.show()

## 13. Side-conditioned calibration

Buy and sell metaorders may show different impact due to sample noise, market drift, or asymmetry.

A robust model should not assume asymmetry unless there is evidence.

In [ ]:
side_calibration_rows = []

for side_name, group in calibration_sample.groupby("side_name"):
    params, _ = calibrate_power_law(group)
    side_calibration_rows.append({
        "side_name": side_name,
        **params,
        "mean_participation": group["participation"].mean(),
        "mean_impact_bps": group["impact_bps"].mean(),
    })

side_calibration = pd.DataFrame(side_calibration_rows)

side_calibration

## 14. Bootstrap confidence intervals

Bootstrap the calibration sample and estimate uncertainty in $Y$ and $\beta$.

In [ ]:
def bootstrap_power_law(sample, config):
    rng = np.random.default_rng(config.seed + 999)
    rows = []
    n = len(sample)

    for b in range(config.bootstrap_samples):
        idx = rng.integers(0, n, size=n)
        boot = sample.iloc[idx]
        try:
            params, _ = calibrate_power_law(boot)
            rows.append({
                "bootstrap_id": b,
                "Y": params["Y"],
                "beta": params["beta"],
                "alpha": params["alpha"],
                "r2": params["r2"],
            })
        except Exception:
            continue

    return pd.DataFrame(rows)

bootstrap_params = bootstrap_power_law(calibration_sample, config)

bootstrap_ci = pd.DataFrame([{
    "Y_mean": bootstrap_params["Y"].mean(),
    "Y_p05": bootstrap_params["Y"].quantile(0.05),
    "Y_p50": bootstrap_params["Y"].quantile(0.50),
    "Y_p95": bootstrap_params["Y"].quantile(0.95),
    "beta_mean": bootstrap_params["beta"].mean(),
    "beta_p05": bootstrap_params["beta"].quantile(0.05),
    "beta_p50": bootstrap_params["beta"].quantile(0.50),
    "beta_p95": bootstrap_params["beta"].quantile(0.95),
}])

bootstrap_ci

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(bootstrap_params["beta"], bins=50)
plt.axvline(0.5, linestyle="--", label="square-root beta=0.5")
plt.title("Bootstrap Distribution of Impact Exponent beta")
plt.xlabel("beta")
plt.ylabel("Count")
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(bootstrap_params["Y"], bins=50)
plt.axvline(config.true_y, linestyle="--", label="base true Y")
plt.title("Bootstrap Distribution of Impact Coefficient Y")
plt.xlabel("Y")
plt.ylabel("Count")
plt.legend()
plt.show()

## 15. Prediction intervals

For a new order, model uncertainty comes from:

1. parameter uncertainty;
2. residual noise;
3. regime uncertainty;
4. volatility and volume estimation error.

Here we approximate parameter uncertainty using bootstrap estimates.

In [ ]:
def predict_impact_distribution(participation, daily_vol, bootstrap_params):
    preds = bootstrap_params["Y"].to_numpy() * daily_vol * participation ** bootstrap_params["beta"].to_numpy()
    return pd.Series(preds)

new_order_participation_grid = np.array([0.005, 0.01, 0.025, 0.05, 0.10, 0.20])
new_order_vol = market_state["daily_vol"].median()

prediction_rows = []
for p in new_order_participation_grid:
    preds = predict_impact_distribution(p, new_order_vol, bootstrap_params)
    prediction_rows.append({
        "participation": p,
        "daily_vol": new_order_vol,
        "predicted_impact_bps_mean": preds.mean() * 10000,
        "predicted_impact_bps_p05": preds.quantile(0.05) * 10000,
        "predicted_impact_bps_p50": preds.quantile(0.50) * 10000,
        "predicted_impact_bps_p95": preds.quantile(0.95) * 10000,
    })

prediction_interval_report = pd.DataFrame(prediction_rows)

prediction_interval_report

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(prediction_interval_report["participation"], prediction_interval_report["predicted_impact_bps_p50"], marker="o", label="median")
plt.fill_between(
    prediction_interval_report["participation"],
    prediction_interval_report["predicted_impact_bps_p05"],
    prediction_interval_report["predicted_impact_bps_p95"],
    alpha=0.25,
    label="5-95% bootstrap interval",
)
plt.xscale("log")
plt.title("Predicted Impact Interval")
plt.xlabel("Participation Q/V")
plt.ylabel("Impact, bps")
plt.legend()
plt.show()

## 16. Intraday volume and schedule impact

A metaorder can be split across intervals.

Interval impact depends on interval participation:

$$
p_k = \frac{q_k}{V_k}
$$

A schedule with the same total $Q$ can have different cost depending on volume curve alignment.

In [ ]:
def intraday_volume_curve(n=20):
    u = np.linspace(0, 1, n)
    curve = 1.0 + 0.45 * np.exp(-((u - 0.05) / 0.18) ** 2) + 0.55 * np.exp(-((u - 0.95) / 0.18) ** 2)
    return curve / curve.sum()

def schedule_weights(n, name):
    if name == "TWAP":
        w = np.ones(n)
    elif name == "FrontLoaded":
        w = np.exp(-np.linspace(0, 4, n))
    elif name == "BackLoaded":
        w = np.exp(np.linspace(0, 4, n))
    elif name == "VWAP":
        w = intraday_volume_curve(n)
    else:
        raise ValueError("Unknown schedule")
    return w / w.sum()

def schedule_impact_cost(total_q, daily_volume, daily_vol, y, beta, n=20, schedule_name="TWAP"):
    volume_curve = intraday_volume_curve(n)
    interval_volume = daily_volume * volume_curve
    weights = schedule_weights(n, schedule_name)
    interval_q = total_q * weights
    participation = interval_q / interval_volume
    impact_return = y * daily_vol * participation ** beta
    # Cost approximation: weighted by shares traded.
    weighted_impact_return = np.sum((interval_q / total_q) * impact_return)
    return {
        "schedule": schedule_name,
        "total_q": total_q,
        "daily_volume": daily_volume,
        "total_participation": total_q / daily_volume,
        "weighted_impact_bps": weighted_impact_return * 10000,
        "max_interval_participation": participation.max(),
        "mean_interval_participation": participation.mean(),
    }

schedule_impact_rows = []
total_participation = 0.10
daily_volume = market_state["adv_shares"].median()
total_q = total_participation * daily_volume
daily_vol = market_state["daily_vol"].median()
y_hat = power_law_params["Y"]
beta_hat = power_law_params["beta"]

for schedule_name in ["TWAP", "VWAP", "FrontLoaded", "BackLoaded"]:
    schedule_impact_rows.append(schedule_impact_cost(total_q, daily_volume, daily_vol, y_hat, beta_hat, n=20, schedule_name=schedule_name))

schedule_impact_report = pd.DataFrame(schedule_impact_rows).sort_values("weighted_impact_bps")

schedule_impact_report

## 17. Transient impact decay

Observed impact may decay over time.

We model post-trade impact path:

$$
I(t) = I_{perm} + I_{temp}e^{-kt}
$$

where:

- $I_{perm}$: permanent component;
- $I_{temp}$: transient component;
- $k$: decay speed.

Half-life:

$$
h = \frac{\ln 2}{k}
$$

In [ ]:
def simulate_impact_decay(metaorders, config, n_intervals=30):
    rng = np.random.default_rng(config.seed + 444)
    rows = []

    decay_k = np.log(2) / config.decay_half_life_intervals

    sample = metaorders.sample(n=min(2_000, len(metaorders)), random_state=config.seed)

    for _, order in sample.iterrows():
        clean = order["clean_abs_impact_return"]
        permanent = config.permanent_fraction * clean
        temporary = clean - permanent

        for h in range(n_intervals + 1):
            impact = permanent + temporary * np.exp(-decay_k * h)
            observed = impact + rng.normal(0.0, order["daily_vol"] * 0.03)
            rows.append({
                "order_id": order["order_id"],
                "horizon": h,
                "clean_impact": clean,
                "permanent_component": permanent,
                "temporary_component": temporary,
                "observed_abs_impact_return": observed,
                "observed_abs_impact_bps": observed * 10000,
            })

    return pd.DataFrame(rows)

impact_decay = simulate_impact_decay(metaorders, config)

decay_summary = (
    impact_decay
    .groupby("horizon")
    .agg(
        mean_observed_impact_bps=("observed_abs_impact_bps", "mean"),
        p25_observed_impact_bps=("observed_abs_impact_bps", lambda x: x.quantile(0.25)),
        p75_observed_impact_bps=("observed_abs_impact_bps", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

decay_summary.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(decay_summary["horizon"], decay_summary["mean_observed_impact_bps"], marker="o", label="mean")
plt.fill_between(
    decay_summary["horizon"],
    decay_summary["p25_observed_impact_bps"],
    decay_summary["p75_observed_impact_bps"],
    alpha=0.25,
    label="25-75%",
)
plt.title("Transient Impact Decay")
plt.xlabel("Intervals after execution")
plt.ylabel("Observed absolute impact, bps")
plt.legend()
plt.show()

## 18. Estimate decay half-life

Fit:

$$
I_h - I_\infty \approx A e^{-kh}
$$

For this synthetic example we use the final horizon as a proxy for $I_\infty$.

In [ ]:
def estimate_decay_half_life(decay_summary):
    df = decay_summary.copy()
    terminal = df["mean_observed_impact_bps"].iloc[-1]
    df["excess_impact"] = df["mean_observed_impact_bps"] - terminal
    df = df[df["excess_impact"] > 0].copy()

    if len(df) < 5:
        return pd.DataFrame([{
            "estimated_decay_k": np.nan,
            "estimated_half_life_intervals": np.nan,
            "terminal_impact_bps_proxy": terminal,
            "fit_note": "insufficient_positive_excess_impact",
        }])

    X = np.column_stack([np.ones(len(df)), -df["horizon"].to_numpy()])
    y = np.log(df["excess_impact"].to_numpy())

    beta = np.linalg.pinv(X.T @ X) @ X.T @ y
    intercept, k = beta

    fitted = X @ beta
    ss_res = ((y - fitted) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    half_life = np.log(2) / k if k > 0 else np.nan

    return pd.DataFrame([{
        "estimated_decay_k": k,
        "estimated_half_life_intervals": half_life,
        "terminal_impact_bps_proxy": terminal,
        "decay_fit_r2": r2,
        "fit_note": "ok",
    }])

decay_half_life_report = estimate_decay_half_life(decay_summary)

decay_half_life_report

## 19. Capacity curves

Using the calibrated model, estimate impact cost as AUM scales.

For a portfolio rebalance with turnover $TO$:

$$
Q = AUM \times TO
$$

Given daily dollar volume $V$, impact is:

$$
I = Y\sigma \Big(\frac{Q}{V}\Big)^\beta
$$

We calculate expected impact for different AUM levels.

In [ ]:
def capacity_curve(y, beta, daily_vol, daily_dollar_volume, turnover):
    aum_grid = np.array([1e6, 2.5e6, 5e6, 10e6, 25e6, 50e6, 100e6, 250e6, 500e6, 1e9, 2e9])
    rows = []

    for aum in aum_grid:
        q_dollar = aum * turnover
        participation = q_dollar / daily_dollar_volume
        impact_return = y * daily_vol * participation ** beta
        impact_cost_dollar = q_dollar * impact_return

        rows.append({
            "aum": aum,
            "turnover": turnover,
            "q_dollar": q_dollar,
            "daily_dollar_volume": daily_dollar_volume,
            "participation": participation,
            "impact_bps": impact_return * 10000,
            "impact_cost_dollar": impact_cost_dollar,
            "impact_cost_bps_of_aum": impact_cost_dollar / aum * 10000,
        })

    return pd.DataFrame(rows)

median_daily_dollar_volume = (market_state["adv_shares"] * market_state["price"]).median()
median_daily_vol = market_state["daily_vol"].median()

capacity_report = capacity_curve(
    y_hat,
    beta_hat,
    median_daily_vol,
    median_daily_dollar_volume,
    turnover=0.10,
)

capacity_report

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(capacity_report["aum"], capacity_report["impact_cost_bps_of_aum"], marker="o")
plt.xscale("log")
plt.title("Capacity Curve: Impact Cost as bps of AUM")
plt.xlabel("AUM")
plt.ylabel("Impact cost, bps of AUM")
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(capacity_report["aum"], capacity_report["participation"], marker="o")
plt.xscale("log")
plt.axhline(0.10, linestyle="--", label="10% participation")
plt.title("Capacity Curve: Participation")
plt.xlabel("AUM")
plt.ylabel("Participation")
plt.legend()
plt.show()

## 20. Stress testing

Stress scenario:

- impact coefficient $Y$ increases;
- volatility increases;
- volume decreases.

This is when capacity usually disappears.

In [ ]:
stress_capacity_report = capacity_curve(
    y_hat * config.stress_y_multiplier,
    beta_hat,
    median_daily_vol * config.stress_vol_multiplier,
    median_daily_dollar_volume * 0.50,
    turnover=0.10,
)

stress_capacity_report["scenario"] = "stress"
capacity_report["scenario"] = "base"

capacity_scenarios = pd.concat([capacity_report, stress_capacity_report], ignore_index=True)

capacity_scenarios.head()

In [ ]:
plt.figure(figsize=(10, 6))
for scenario, group in capacity_scenarios.groupby("scenario"):
    plt.plot(group["aum"], group["impact_cost_bps_of_aum"], marker="o", label=scenario)
plt.xscale("log")
plt.title("Base vs Stress Capacity Impact")
plt.xlabel("AUM")
plt.ylabel("Impact cost, bps of AUM")
plt.legend()
plt.show()

## 21. Model error diagnostics

We examine residuals by:

- participation bucket;
- regime;
- side;
- volatility bucket.

A good impact model should not have systematic residual bias.

In [ ]:
def model_residual_diagnostics(sample, y_hat, beta_hat):
    df = sample.copy()
    df["predicted_normalised_impact"] = y_hat * df["participation"] ** beta_hat
    df["residual_normalised_impact"] = df["normalised_abs_impact"] - df["predicted_normalised_impact"]
    df["residual_bps"] = df["residual_normalised_impact"] * df["daily_vol"] * 10000

    df["participation_bucket"] = pd.qcut(df["participation"], q=5, duplicates="drop")
    df["vol_bucket"] = pd.qcut(df["daily_vol"], q=5, duplicates="drop")

    by_participation = (
        df.groupby("participation_bucket", observed=False)
        .agg(
            n=("order_id", "count"),
            mean_residual_bps=("residual_bps", "mean"),
            median_residual_bps=("residual_bps", "median"),
            rmse_bps=("residual_bps", lambda x: np.sqrt(np.mean(x**2))),
        )
        .reset_index()
    )

    by_regime = (
        df.groupby("regime_name")
        .agg(
            n=("order_id", "count"),
            mean_residual_bps=("residual_bps", "mean"),
            median_residual_bps=("residual_bps", "median"),
            rmse_bps=("residual_bps", lambda x: np.sqrt(np.mean(x**2))),
        )
        .reset_index()
    )

    by_side = (
        df.groupby("side_name")
        .agg(
            n=("order_id", "count"),
            mean_residual_bps=("residual_bps", "mean"),
            median_residual_bps=("residual_bps", "median"),
            rmse_bps=("residual_bps", lambda x: np.sqrt(np.mean(x**2))),
        )
        .reset_index()
    )

    return df, by_participation, by_regime, by_side

residuals, residual_by_participation, residual_by_regime, residual_by_side = model_residual_diagnostics(
    calibration_sample,
    y_hat,
    beta_hat,
)

residual_by_participation, residual_by_regime, residual_by_side

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(residuals["participation"], residuals["residual_bps"], alpha=0.25)
plt.xscale("log")
plt.axhline(0, linestyle="--")
plt.title("Residuals vs Participation")
plt.xlabel("Participation Q/V")
plt.ylabel("Residual, bps")
plt.show()

## 22. Governance flags

An impact model should be reviewed if:

1. estimated $\beta$ is far from expected range;
2. confidence interval is too wide;
3. regime coefficient differs strongly;
4. residual bias is large;
5. stress capacity cost is too high;
6. calibration sample is too small;
7. impact parameters are not calibrated from real fills.

In [ ]:
beta_hat = power_law_params["beta"]
y_hat = power_law_params["Y"]
beta_ci_width = bootstrap_ci["beta_p95"].iloc[0] - bootstrap_ci["beta_p05"].iloc[0]
y_ci_width = bootstrap_ci["Y_p95"].iloc[0] - bootstrap_ci["Y_p05"].iloc[0]

regime_y_ratio = (
    regime_calibration["Y"].max()
    / max(regime_calibration["Y"].min(), 1e-12)
)

max_abs_residual_bias = max(
    abs(residual_by_participation["mean_residual_bps"]).max(),
    abs(residual_by_regime["mean_residual_bps"]).max(),
    abs(residual_by_side["mean_residual_bps"]).max(),
)

stress_100m_cost = stress_capacity_report.loc[stress_capacity_report["aum"] == 100_000_000, "impact_cost_bps_of_aum"]
stress_100m_cost_value = float(stress_100m_cost.iloc[0]) if len(stress_100m_cost) else np.nan

governance_flags = pd.DataFrame([{
    "estimated_Y": y_hat,
    "estimated_beta": beta_hat,
    "beta_ci_width": beta_ci_width,
    "Y_ci_width": y_ci_width,
    "regime_Y_ratio": regime_y_ratio,
    "max_abs_residual_bias_bps": max_abs_residual_bias,
    "stress_100m_impact_cost_bps_of_aum": stress_100m_cost_value,
    "calibration_sample_size": len(calibration_sample),
    "flag_beta_far_from_square_root": bool(abs(beta_hat - 0.5) > 0.20),
    "flag_beta_uncertainty_wide": bool(beta_ci_width > 0.30),
    "flag_regime_difference_large": bool(regime_y_ratio > 2.5),
    "flag_residual_bias_large": bool(max_abs_residual_bias > 5.0),
    "flag_stress_capacity_cost_high": bool(stress_100m_cost_value > 50.0) if np.isfinite(stress_100m_cost_value) else False,
    "flag_small_calibration_sample": bool(len(calibration_sample) < 500),
    "flag_not_real_fill_data": True,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_beta_far_from_square_root",
        "flag_beta_uncertainty_wide",
        "flag_regime_difference_large",
        "flag_residual_bias_large",
        "flag_stress_capacity_cost_high",
        "flag_small_calibration_sample",
        "flag_not_real_fill_data",
    ]
].any(axis=1)

governance_flags

## 23. Audit checks

Numerical and process checks:

1. participation is positive;
2. calibration sample is non-empty;
3. model parameters are finite;
4. bootstrap estimates are finite;
5. prediction intervals are ordered;
6. capacity costs increase with AUM;
7. decay half-life is finite;
8. governance flags exist.

In [ ]:
audit_rows = []

audit_rows.append({
    "check": "participation_positive",
    "value": bool((metaorders["participation"] > 0).all()),
    "passed": bool((metaorders["participation"] > 0).all()),
})

audit_rows.append({
    "check": "calibration_sample_nonempty",
    "value": int(len(calibration_sample)),
    "passed": bool(len(calibration_sample) > 0),
})

params_finite = bool(np.isfinite([power_law_params["Y"], power_law_params["beta"], power_law_params["r2"]]).all())
audit_rows.append({
    "check": "power_law_parameters_finite",
    "value": params_finite,
    "passed": params_finite,
})

bootstrap_finite = bool(np.isfinite(bootstrap_params[["Y", "beta"]].to_numpy()).all())
audit_rows.append({
    "check": "bootstrap_parameters_finite",
    "value": bootstrap_finite,
    "passed": bootstrap_finite,
})

prediction_ordered = bool(
    (
        prediction_interval_report["predicted_impact_bps_p05"]
        <= prediction_interval_report["predicted_impact_bps_p50"]
    ).all()
    and (
        prediction_interval_report["predicted_impact_bps_p50"]
        <= prediction_interval_report["predicted_impact_bps_p95"]
    ).all()
)
audit_rows.append({
    "check": "prediction_intervals_ordered",
    "value": prediction_ordered,
    "passed": prediction_ordered,
})

capacity_monotone = bool(capacity_report["impact_cost_bps_of_aum"].is_monotonic_increasing)
audit_rows.append({
    "check": "capacity_cost_increases_with_aum",
    "value": capacity_monotone,
    "passed": capacity_monotone,
})

half_life_finite = bool(np.isfinite(decay_half_life_report["estimated_half_life_intervals"].iloc[0]))
audit_rows.append({
    "check": "decay_half_life_finite",
    "value": half_life_finite,
    "passed": half_life_finite,
})

governance_exists = bool(len(governance_flags) == 1)
audit_rows.append({
    "check": "governance_flags_exist",
    "value": governance_exists,
    "passed": governance_exists,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 24. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

market_state.to_csv(output_dir / "market_state.csv")
metaorders.to_csv(output_dir / "metaorders.csv", index=False)
calibration_sample.to_csv(output_dir / "calibration_sample.csv", index=False)
calibration_report.to_csv(output_dir / "calibration_report.csv", index=False)
pd.Series(power_law_params).to_frame("value").to_csv(output_dir / "power_law_params.csv")
model_comparison.to_csv(output_dir / "model_comparison.csv", index=False)
oos_model_comparison.to_csv(output_dir / "oos_model_comparison.csv", index=False)
regime_calibration.to_csv(output_dir / "regime_calibration.csv", index=False)
side_calibration.to_csv(output_dir / "side_calibration.csv", index=False)
bootstrap_params.to_csv(output_dir / "bootstrap_params.csv", index=False)
bootstrap_ci.to_csv(output_dir / "bootstrap_confidence_intervals.csv", index=False)
prediction_interval_report.to_csv(output_dir / "prediction_interval_report.csv", index=False)
schedule_impact_report.to_csv(output_dir / "schedule_impact_report.csv", index=False)
impact_decay.to_csv(output_dir / "impact_decay_paths.csv", index=False)
decay_summary.to_csv(output_dir / "decay_summary.csv", index=False)
decay_half_life_report.to_csv(output_dir / "decay_half_life_report.csv", index=False)
capacity_report.to_csv(output_dir / "capacity_report_base.csv", index=False)
stress_capacity_report.to_csv(output_dir / "capacity_report_stress.csv", index=False)
capacity_scenarios.to_csv(output_dir / "capacity_scenarios.csv", index=False)
residuals.to_csv(output_dir / "model_residuals.csv", index=False)
residual_by_participation.to_csv(output_dir / "residual_by_participation.csv", index=False)
residual_by_regime.to_csv(output_dir / "residual_by_regime.csv", index=False)
residual_by_side.to_csv(output_dir / "residual_by_side.csv", index=False)
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

manifest = {
    "dataset_name": "square_root_market_impact_model_outputs",
    "schema_version": "square_root_market_impact_model_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "estimated_power_law": power_law_params,
    "bootstrap_ci": bootstrap_ci.to_dict(orient="records"),
    "model_comparison": model_comparison.to_dict(orient="records"),
    "oos_model_comparison": oos_model_comparison.to_dict(orient="records"),
    "regime_calibration": regime_calibration.to_dict(orient="records"),
    "decay_half_life_report": decay_half_life_report.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook calibrates a square-root / power-law market impact model from synthetic metaorders.",
        "The main model is I/sigma = Y * (Q/V)^beta.",
        "Linear, square-root and free power-law specifications are compared.",
        "Regime-conditioned, side-conditioned and out-of-sample diagnostics are included.",
        "Transient impact decay and permanent impact proxy are simulated.",
        "Capacity curves translate impact into cost as AUM scales.",
        "The data is synthetic and should be replaced with real fills or metaorder records before production use."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

output_dir / "power_law_params.csv", output_dir / "model_comparison.csv", output_dir / "capacity_scenarios.csv", output_dir / "governance_flags.csv", manifest_path

## 25. Practical checklist for square-root impact modelling

Before using a square-root impact model:

1. **Define impact measurement horizon.**
2. **Separate spread, temporary impact and permanent impact.**
3. **Normalise by volatility.**
4. **Normalise size by volume.**
5. **Estimate the exponent $\beta$, not just $Y$.**
6. **Check regimes separately.**
7. **Check buy/sell symmetry.**
8. **Bootstrap parameter uncertainty.**
9. **Inspect residuals by participation and volatility.**
10. **Stress $Y$, volatility and volume before capacity approval.**

## 26. Limitations

### Synthetic metaorders

The notebook uses synthetic metaorder data. Real calibration needs parent-order records or carefully reconstructed metaorders.

### Measurement horizon

Impact depends strongly on whether it is measured at arrival, VWAP, end-of-order, close, or post-trade horizons.

### Directional alpha

If orders are correlated with alpha, observed price moves mix impact and information.

### Regime instability

Impact coefficients vary by market, asset, liquidity, volatility, venue and time.

### Log filtering bias

Log-log calibration drops non-positive observed impacts, which can bias estimates.

### No order book

The model does not use queue position, depth, or L1/L2 microstructure.

### No causal identification

Observed impact is not automatically causal without careful controls.

## 27. Research frontier and extensions

Important extensions include:

1. causal impact estimation;
2. transient propagator models;
3. nonlinear temporary impact calibration;
4. order-book-depth-aware impact;
5. metaorder reconstruction;
6. impact decay by regime;
7. cross-impact between correlated assets;
8. futures-specific impact with tick size and price limits;
9. L1 bid/ask impact proxy from top-of-book data;
10. live TCA calibration loop.

## 28. Suggested follow-up notebooks

This notebook naturally leads to:

1. **06_04_execution_algorithms_twap_vwap_pov**  
   Use the impact model to compare execution algorithms.

2. **06_05_order_management_system_simulator**  
   Track orders, child orders and fills.

3. **06_07_latency_and_queue_position_model**  
   Move from parent-order impact to microstructure fill risk.

4. **06_11_l1_bid_ask_backtest_execution_model**  
   Apply L1 bid/ask data to estimate realised execution costs.

5. **06_13_execution_research_capstone**  
   Combine optimal execution, impact and execution risk controls.

## 29. Summary

This notebook implemented square-root market impact modelling.

It showed how to:

1. simulate daily market liquidity states;
2. generate synthetic metaorders;
3. measure observed and clean impact;
4. prepare a calibration sample;
5. calibrate a log-log power law;
6. estimate the impact exponent $\beta$;
7. compare linear, square-root and free power-law models;
8. evaluate out-of-sample fit;
9. calibrate impact by liquidity regime;
10. calibrate impact by side;
11. bootstrap confidence intervals;
12. create prediction intervals;
13. evaluate schedule-level impact;
14. simulate transient impact decay;
15. estimate decay half-life;
16. build capacity curves;
17. stress impact under high-vol / low-liquidity conditions;
18. analyse residual bias;
19. create governance flags and audit checks;
20. save outputs and manifests.

The key computational takeaway:

> The square-root law can be estimated as a log-log regression of normalised impact on participation.

The key financial takeaway:

> Capacity is not about average turnover alone; it is about how turnover interacts with volatility, volume, impact and stress liquidity.

## 30. Further reading

- Almgren et al. on market impact and execution.
- Bouchaud, Farmer and Lillo on empirical market impact.
- Gatheral and Schied on market impact models.
- Tóth et al. on the square-root impact law.
- Cartea, Jaimungal and Penalva, *Algorithmic and High-Frequency Trading.*
- Kissell, *The Science of Algorithmic Trading and Portfolio Management.*